In [0]:
from pyspark.sql.functions import explode, col, lit, when
from delta.tables import DeltaTable
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

dbutils.widgets.text("player_id", "") 
player_id = dbutils.widgets.get("player_id")
logger.info("id from adf: " + player_id)
df = spark.read.json("abfss://faceit@storageonline23.dfs.core.windows.net/players")

matches = df.select(explode("items").alias("match"))

f1 = matches.select(
    col("match.match_id"),
    lit("faction1").alias("faction"),
    explode("match.teams.faction1.players").alias("player"),
    col("match.results.winner").alias("winner"),
    col("match.results.score.faction1").alias("team_score"),
    col("match.results.score.faction2").alias("opponent_score")
)
f2 = matches.select(
    col("match.match_id"),
    lit("faction2").alias("faction"),
    explode("match.teams.faction2.players").alias("player"),
    col("match.results.winner").alias("winner"),
    col("match.results.score.faction2").alias("team_score"),
    col("match.results.score.faction1").alias("opponent_score")
)

players = f1.unionByName(f2)

players = players.withColumn(
    "outcome",
    when(col("faction") == col("winner"), "Win").otherwise("Lose")
)

result = players.select(
    "match_id",
    col("player.player_id").alias("player_id"),
    "outcome",
    "team_score",
    "opponent_score"
)

delta_table = DeltaTable.forName(spark, "ddca26online23.default.silver_player_history")

logger.info(f"Target rows before merge: {delta_table.toDF().count()}")

delta_table.alias("tgt").merge(
    result.alias("src"),
    "tgt.match_id = src.match_id AND tgt.player_id = src.player_id"
).whenNotMatchedInsertAll().execute()

logger.info(f"Target rows after merge: {delta_table.toDF().count()}")
